In [0]:
# Bronze Layer: Raw Payments Ingestion (JSON Format)
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, col

spark = SparkSession.builder.getOrCreate()

LANDING_PATH  = "s3://ecom-landing-zone-2026/landing/payments/"
CHECKPOINT    = "s3://ecom-landing-zone-2026/checkpoints/payments_checkpoint/"
BRONZE_TABLE  = "ecomstore.ecomlake.bronze_payments"
SOURCE_SYSTEM = "payment_gateway"


spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
        payment_id STRING, order_id STRING, payment_method STRING, payment_status STRING, 
        amount STRING, transaction_id STRING, payment_date STRING, gateway STRING, 
        created_at STRING, updated_at STRING, _batch_id STRING, _rescued_data STRING,
        _ingestion_timestamp TIMESTAMP, _source_file_name STRING, 
        _pipeline_layer STRING, _source_system_override STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact' = 'true'
    )
""")

payments_schema = """
    payment_id STRING, order_id STRING, payment_method STRING, payment_status STRING, 
    amount STRING, transaction_id STRING, payment_date STRING, gateway STRING, 
    created_at STRING, updated_at STRING, _batch_id STRING
"""

# 2. Auto Loader Read with Rescued Data Catch
raw_payments_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT + "schema/")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaHints", payments_schema)
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")    # Traps corrupted JSON
    .option("multiLine", "false")
    .load(LANDING_PATH)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file_name", col("_metadata.file_path"))
    .withColumn("_pipeline_layer", lit("bronze"))
    .withColumn("_source_system_override", lit(SOURCE_SYSTEM))
)

# 3. Write to Bronze
(
    raw_payments_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)
print(f"✅ Bronze ingestion complete for: {BRONZE_TABLE}")